# Aggregate Functions

## Overview
Aggregate functions collapse a set of rows into a single value. They are used inside `SELECT` with `GROUP BY`, or without `GROUP BY` to aggregate the entire table.

**Core aggregate functions — supported in SQLite and PostgreSQL:**

| Function | Returns | NULL behaviour |
|---|---|---|
| `COUNT(*)` | Number of rows in the group | Counts all rows, including NULLs |
| `COUNT(col)` | Number of non-NULL values | Skips NULLs |
| `COUNT(DISTINCT col)` | Number of distinct non-NULL values | Skips NULLs |
| `SUM(col)` | Total of numeric values | Skips NULLs; returns NULL if all are NULL |
| `AVG(col)` | Arithmetic mean | Skips NULLs — denominator is non-NULL count |
| `MIN(col)` | Smallest value | Skips NULLs; works on text, dates, numbers |
| `MAX(col)` | Largest value | Skips NULLs; works on text, dates, numbers |

**PostgreSQL extras:** `STDDEV()`, `VARIANCE()`, `PERCENTILE_CONT()`, `PERCENTILE_DISC()`, `MODE()`, `CORR()`, `BOOL_AND()`, `BOOL_OR()`, `ARRAY_AGG()`, `JSON_AGG()`

**Key insight:** Every aggregate function except `COUNT(*)` ignores NULL values. This can silently make averages look higher or totals look lower than they should be if NULLs represent missing data rather than "not applicable."

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    segment TEXT, province TEXT
);
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY, customer_id INTEGER, account_type TEXT,
    province TEXT, opened_date TEXT, status TEXT
);
CREATE TABLE transactions (
    txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT,
    txn_type TEXT, amount REAL, category TEXT, flagged INTEGER
);
CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY, customer_id INTEGER, loan_type TEXT,
    principal REAL, rate_pct REAL, term_months INTEGER,
    issued_date TEXT, status TEXT
);
INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');
INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),
  (102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),
  (104,2,'Investment','BC','2021-01-10','Active'),
  (105,3,'Chequing','ON','2018-05-20','Active'),
  (106,3,'Investment','ON','2018-05-20','Active'),
  (107,3,'Savings','ON','2022-11-01','Active'),
  (108,4,'Chequing','NB','2015-09-30','Active'),
  (109,4,'Loan','NB','2020-04-01','Closed'),
  (110,5,'Chequing','BC','2021-06-15','Active'),
  (111,5,'Savings','BC','2021-06-15','Suspended'),
  (112,6,'Chequing','AB','2017-11-22','Active'),
  (113,7,'Investment','ON','2016-03-08','Active'),
  (114,7,'Chequing','ON','2016-03-08','Active'),
  (115,8,'Chequing','QC','2023-01-05','Active'),
  (116,9,'Chequing','BC','2022-08-19','Active'),
  (117,9,'Investment','BC','2022-08-19','Active'),
  (118,10,'Chequing','ON','2020-12-01','Active'),
  (119,10,'Savings','ON','2020-12-01','Active');
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),
  (1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),
  (1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),
  (1006,102,'2023-01-15','Deposit',500.00,'Transfer',0),
  (1007,102,'2023-03-10','Interest',12.50,'Interest',0),
  (1008,103,'2023-01-08','Deposit',15000.00,'Payroll',0),
  (1009,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1010,103,'2023-02-08','Deposit',15000.00,'Payroll',0),
  (1011,104,'2023-01-31','Deposit',5000.00,'Transfer',0),
  (1012,105,'2023-01-10','Deposit',22000.00,'Payroll',0),
  (1013,105,'2023-01-28','Withdrawal',-8500.00,'Rent',0),
  (1014,105,'2023-02-10','Deposit',22000.00,'Payroll',0),
  (1015,106,'2023-02-01','Deposit',50000.00,'Investment',0),
  (1016,108,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1017,108,'2023-01-19','Withdrawal',-700.00,'Rent',0),
  (1018,108,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1019,110,'2023-01-15','Deposit',8500.00,'Payroll',0),
  (1020,110,'2023-02-01','Withdrawal',-2100.00,'Rent',0),
  (1021,112,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1022,112,'2023-02-10','Fee',-25.00,'Fee',1),
  (1023,113,'2023-01-05','Deposit',75000.00,'Investment',0),
  (1024,114,'2023-01-12','Deposit',12000.00,'Payroll',0),
  (1025,114,'2023-02-12','Deposit',12000.00,'Payroll',0),
  (1026,115,'2023-01-10','Deposit',2800.00,'Payroll',0),
  (1027,115,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1028,116,'2023-01-18','Deposit',9200.00,'Payroll',0),
  (1029,117,'2023-02-05','Deposit',10000.00,'Investment',0),
  (1030,118,'2023-01-22','Deposit',4500.00,'Payroll',0),
  (1031,118,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1032,119,'2023-02-20','Interest',18.75,'Interest',0),
  (1033,101,'2023-03-05','Deposit',4200.00,'Payroll',NULL),
  (1034,103,'2023-03-08','Deposit',15000.00,'Payroll',0),
  (1035,112,'2023-03-15','Withdrawal',-450.00,'Groceries',1);
INSERT INTO loans VALUES
  (201,1,'Personal',15000.00,7.5,36,'2021-06-01','Current'),
  (202,2,'Mortgage',480000.00,4.2,300,'2020-07-15','Current'),
  (203,3,'HELOC',100000.00,6.1,60,'2019-02-28','Current'),
  (204,4,'Auto',28000.00,5.9,60,'2020-04-01','Paid Off'),
  (205,5,'Mortgage',390000.00,4.8,300,'2021-06-20','Current'),
  (206,6,'Personal',8000.00,9.2,24,'2022-03-10','Delinquent'),
  (207,7,'Mortgage',750000.00,3.9,300,'2018-09-01','Current'),
  (208,8,'Auto',22000.00,6.5,48,'2023-01-15','Current'),
  (209,9,'Personal',12000.00,8.1,36,'2022-08-25','Current'),
  (210,10,'Auto',35000.00,5.4,60,'2021-03-01','Current'),
  (211,3,'Personal',25000.00,6.8,48,'2020-11-15','Paid Off'),
  (212,7,'HELOC',200000.00,5.5,60,'2022-06-01','Current');
""")
conn.commit()
print('Finance schema ready.')


Finance schema ready.


---
## COUNT variants — the most important distinctions

In [2]:
# COUNT(*) vs COUNT(col) vs COUNT(DISTINCT col)
print('=== COUNT variants on the transactions table ===')
q = """
SELECT
    COUNT(*)                    AS count_all_rows,
    COUNT(flagged)              AS count_flagged_not_null,
    COUNT(DISTINCT flagged)     AS count_distinct_flagged_values,
    COUNT(category)             AS count_category_not_null,
    COUNT(DISTINCT category)    AS count_distinct_categories,
    COUNT(DISTINCT account_id)  AS count_distinct_accounts
FROM transactions
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Demonstrate with a focused example
print()
print('=== Why COUNT(*) vs COUNT(col) differs ===')
print('Transactions where flagged IS NULL:',
      conn.execute("SELECT COUNT(*) FROM transactions WHERE flagged IS NULL").fetchone()[0])
print('COUNT(*) includes those rows; COUNT(flagged) does not.')

# Per-account COUNT comparison
print()
print('=== Per-account: COUNT(*) vs COUNT(category) ===')
q2 = """
SELECT  account_id,
        COUNT(*)             AS all_txns,
        COUNT(category)      AS txns_with_category,
        COUNT(*) - COUNT(category) AS missing_category
FROM    transactions
GROUP BY account_id
HAVING  COUNT(*) - COUNT(category) > 0   -- only accounts with missing categories
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== COUNT variants on the transactions table ===
 count_all_rows  count_flagged_not_null  count_distinct_flagged_values  count_category_not_null  count_distinct_categories  count_distinct_accounts
             35                      34                              2                       35                          8                       16

=== Why COUNT(*) vs COUNT(col) differs ===
Transactions where flagged IS NULL: 1
COUNT(*) includes those rows; COUNT(flagged) does not.

=== Per-account: COUNT(*) vs COUNT(category) ===
Empty DataFrame
Columns: [account_id, all_txns, txns_with_category, missing_category]
Index: []


---
## SUM, AVG, MIN, MAX — with NULL awareness

In [3]:
# Basic aggregates on transaction amounts
print('=== Transaction amount aggregates by type ===')
q = """
SELECT  txn_type,
        COUNT(*)                       AS txn_count,
        ROUND(SUM(amount), 2)          AS total,
        ROUND(AVG(amount), 2)          AS average,
        ROUND(MIN(amount), 2)          AS minimum,
        ROUND(MAX(amount), 2)          AS maximum,
        ROUND(MAX(amount) - MIN(amount), 2) AS range_spread
FROM    transactions
GROUP BY txn_type
ORDER BY ABS(total) DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# AVG vs manual SUM/COUNT — critical when NULLs are present
print()
print('=== AVG vs SUM/COUNT — they agree when no NULLs in the column ===')
q2 = """
SELECT
    AVG(amount)              AS avg_builtin,
    ROUND(SUM(amount) * 1.0 / COUNT(amount), 2)  AS avg_manual_excl_null,
    ROUND(SUM(amount) * 1.0 / COUNT(*),     2)   AS avg_manual_incl_null_denom
FROM transactions
WHERE txn_type = 'Deposit'
"""
print(pd.read_sql(q2, conn).to_string(index=False))
print('All three agree here because amount has no NULLs in Deposit rows.')
print('If amount had NULLs, AVG() and avg_manual_excl_null would agree,')
print('but avg_manual_incl_null_denom would give a lower (incorrect) average.')


=== Transaction amount aggregates by type ===
  txn_type  txn_count     total  average  minimum  maximum  range_spread
   Deposit         22 301100.00 13686.36    500.0 75000.00      74500.00
Withdrawal         10 -18650.00 -1865.00  -8500.0  -120.00       8380.00
  Interest          2     31.25    15.63     12.5    18.75          6.25
       Fee          1    -25.00   -25.00    -25.0   -25.00          0.00

=== AVG vs SUM/COUNT — they agree when no NULLs in the column ===
 avg_builtin  avg_manual_excl_null  avg_manual_incl_null_denom
13686.363636              13686.36                    13686.36
All three agree here because amount has no NULLs in Deposit rows.
If amount had NULLs, AVG() and avg_manual_excl_null would agree,
but avg_manual_incl_null_denom would give a lower (incorrect) average.


---
## MIN and MAX on non-numeric columns

In [4]:
# MIN/MAX work on text (alphabetical) and date strings (ISO chronological)
print('=== MIN/MAX on dates and text ===')
q = """
SELECT
    MIN(txn_date)          AS earliest_txn,
    MAX(txn_date)          AS latest_txn,
    MIN(category)          AS first_category_alpha,
    MAX(category)          AS last_category_alpha
FROM transactions
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Per-customer: earliest and latest account opened
print()
print('=== Account date range per customer ===')
q2 = """
SELECT  c.first_name || ' ' || c.last_name  AS customer,
        c.segment,
        COUNT(a.account_id)                  AS accounts,
        MIN(a.opened_date)                   AS first_account,
        MAX(a.opened_date)                   AS latest_account
FROM    customers AS c
INNER JOIN accounts AS a ON c.customer_id = a.customer_id
GROUP BY c.customer_id, c.first_name, c.last_name, c.segment
ORDER BY accounts DESC, first_account
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== MIN/MAX on dates and text ===
earliest_txn latest_txn first_category_alpha last_category_alpha
  2023-01-05 2023-03-15                  Fee           Utilities

=== Account date range per customer ===
        customer segment  accounts first_account latest_account
Fatima Al-Rashid  Wealth         3    2018-05-20     2022-11-01
   James MacLeod  Retail         2    2015-09-30     2020-04-01
       Mei Zhang  Wealth         2    2016-03-08     2016-03-08
     Aroha Ngata  Retail         2    2019-03-15     2019-03-15
       Liam Chen     SME         2    2020-07-01     2021-01-10
   Marcus Okafor  Retail         2    2020-12-01     2020-12-01
    Sofia Petrov     SME         2    2021-06-15     2021-06-15
      Priya Nair     SME         2    2022-08-19     2022-08-19
   Noah Williams  Retail         1    2017-11-22     2017-11-22
  Ethan Tremblay  Retail         1    2023-01-05     2023-01-05


---
## Statistical aggregates and PostgreSQL extensions

In [5]:
# Loan portfolio statistics
print('=== Loan principal statistics by type ===')
q = """
SELECT  loan_type,
        COUNT(*)                       AS loans,
        ROUND(SUM(principal), 2)       AS total_principal,
        ROUND(AVG(principal), 2)       AS avg_principal,
        ROUND(MIN(principal), 2)       AS min_principal,
        ROUND(MAX(principal), 2)       AS max_principal,
        ROUND(AVG(rate_pct), 3)        AS avg_rate_pct,
        MIN(rate_pct)                  AS min_rate,
        MAX(rate_pct)                  AS max_rate
FROM    loans
GROUP BY loan_type
ORDER BY total_principal DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

print()
print('=== PostgreSQL-only statistical aggregates (reference) ===')
print("""
SELECT
    loan_type,
    ROUND(AVG(principal), 2)                              AS mean,
    ROUND(STDDEV(principal), 2)                           AS std_dev,
    ROUND(VARIANCE(principal), 2)                         AS variance,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY principal) AS median,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY principal) AS p25,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY principal) AS p75,
    MODE() WITHIN GROUP (ORDER BY rate_pct)               AS modal_rate
FROM loans
GROUP BY loan_type;

-- CORR() for correlation between two columns:
SELECT CORR(principal, rate_pct) AS principal_rate_correlation FROM loans;
""")

conn.close()


=== Loan principal statistics by type ===
loan_type  loans  total_principal  avg_principal  min_principal  max_principal  avg_rate_pct  min_rate  max_rate
 Mortgage      3        1620000.0      540000.00       390000.0       750000.0         4.300       3.9       4.8
    HELOC      2         300000.0      150000.00       100000.0       200000.0         5.800       5.5       6.1
     Auto      3          85000.0       28333.33        22000.0        35000.0         5.933       5.4       6.5
 Personal      4          60000.0       15000.00         8000.0        25000.0         7.900       6.8       9.2

=== PostgreSQL-only statistical aggregates (reference) ===

SELECT
    loan_type,
    ROUND(AVG(principal), 2)                              AS mean,
    ROUND(STDDEV(principal), 2)                           AS std_dev,
    ROUND(VARIANCE(principal), 2)                         AS variance,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY principal) AS median,
    PERCENTILE_CONT(0.25) WITHIN

---
## Common Pitfalls

**1. AVG silently excludes NULLs from both numerator and denominator**
`AVG(score)` over 10 rows where 3 are NULL computes the average of the 7 non-NULL values — the denominator is 7, not 10. If NULLs represent "zero" or "not attempted," use `AVG(COALESCE(score, 0))` to include them.

**2. SUM returns NULL (not 0) when all values in the group are NULL**
`SUM(amount)` over a group where every amount is NULL returns NULL. Wrap with `COALESCE(SUM(amount), 0)` when you need 0 in reports. This is especially common after LEFT JOINs where unmatched groups have all-NULL columns.

**3. COUNT(col) vs COUNT(*) produces different results when col has NULLs**
This is the most frequently misunderstood aggregate distinction. `COUNT(*)` counts rows; `COUNT(col)` counts non-NULL values of that column. They are the same only when the column has no NULLs. Always be explicit about which you need.

**4. AVG of integers performs integer division in some databases**
In PostgreSQL, `AVG(integer_col)` returns a `NUMERIC` result (correct). In SQLite, it returns a REAL. However, computing `SUM(col) / COUNT(col)` manually on integer columns in SQLite performs integer division, losing the decimal. Use `SUM(col) * 1.0 / COUNT(col)` for manual averages.

**5. MIN/MAX on text follows lexicographic (alphabetical) order, not numeric**
`MAX(amount_as_text)` on `['9', '88', '100']` returns `'9'` because `'9' > '88' > '100'` lexicographically. Always CAST to a numeric type before applying MIN/MAX to values stored as text.

**6. STDDEV and PERCENTILE functions are PostgreSQL-only**
`STDDEV()`, `VARIANCE()`, `PERCENTILE_CONT()`, and `MODE()` do not exist in SQLite. For statistical summaries in SQLite, pull the data into pandas and use `df.describe()` or `df.quantile()`.


---
*sql_methods_library - Samantha McGarrigle*